In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import torch
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import os

def strip_html(text: str) -> str:
    text = str(text)
    # remove tags
    text = re.sub(r"<[^>]+>", " ", text)
    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

def to_binary_label(label_0_4: int) -> int:
    # official mapping from paper/spec
    return 0 if int(label_0_4) in (0, 1) else 1

In [2]:
DATA_TSV = "/kaggle/input/datasets/temirlansergazin/dataset/dontpatronizeme_pcl.tsv"

df = pd.read_csv(
    DATA_TSV,
    sep="\t",
    header=None,
    names=["par_id", "art_id", "keyword", "country", "text", "label"],
    engine="python",
    quoting=3,
    on_bad_lines="skip"
)

# Coerce numeric fields
df["par_id"] = pd.to_numeric(df["par_id"], errors="coerce")
df["label"]  = pd.to_numeric(df["label"], errors="coerce")

# Count issues before fixing
print("NaN par_id:", df["par_id"].isna().sum())
print("NaN label:", df["label"].isna().sum())
print("NaN text:", df["text"].isna().sum())

# Fill text if missing
df["text"] = df["text"].fillna("")

# Remove rows where par_id or label failed numeric parsing
df = df[df["par_id"].notna()]
df = df[df["label"].notna()]

df["par_id"] = df["par_id"].astype(int)
df["label"]  = df["label"].astype(int)

# Clean text
df["text"] = df["text"].apply(strip_html)

# Binary mapping
df["label_bin"] = df["label"].apply(to_binary_label)

print("Final shape:", df.shape)

NaN par_id: 3
NaN label: 3
NaN text: 4
Final shape: (10469, 7)


In [3]:
TRAIN_SPLIT = "/kaggle/input/datasets/temirlansergazin/dataset/train_semeval_parids-labels.csv"
DEV_SPLIT   = "/kaggle/input/datasets/temirlansergazin/dataset/dev_semeval_parids-labels.csv"

train_ids = pd.read_csv(TRAIN_SPLIT)["par_id"].astype(int).tolist()
dev_ids   = pd.read_csv(DEV_SPLIT)["par_id"].astype(int).tolist()

train_df = df[df["par_id"].isin(train_ids)][["par_id", "text", "label_bin"]].copy()
dev_df   = df[df["par_id"].isin(dev_ids)][["par_id", "text", "label_bin"]].copy()

print("train:", train_df.shape, "dev:", dev_df.shape)
train_df["label_bin"].value_counts(), dev_df["label_bin"].value_counts()

train: (8375, 3) dev: (2094, 3)


(label_bin
 0    7581
 1     794
 Name: count, dtype: int64,
 label_bin
 0    1895
 1     199
 Name: count, dtype: int64)

In [4]:
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
dev_ds   = Dataset.from_pandas(dev_df.reset_index(drop=True))

train_ds, dev_ds

(Dataset({
     features: ['par_id', 'text', 'label_bin'],
     num_rows: 8375
 }),
 Dataset({
     features: ['par_id', 'text', 'label_bin'],
     num_rows: 2094
 }))

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "roberta-base"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

train_tok = train_ds.map(tokenize_batch, batched=True)
dev_tok   = dev_ds.map(tokenize_batch, batched=True)

cols_to_remove = ["text"]
train_tok = train_tok.remove_columns([c for c in cols_to_remove if c in train_tok.column_names])
dev_tok   = dev_tok.remove_columns([c for c in cols_to_remove if c in dev_tok.column_names])

train_tok = train_tok.rename_column("label_bin", "labels")
dev_tok   = dev_tok.rename_column("label_bin", "labels")

train_tok.set_format("torch")
dev_tok.set_format("torch")

# sanity check labels
batch = next(iter(torch.utils.data.DataLoader(train_tok, batch_size=64)))
print("Label dtype:", batch["labels"].dtype)
print("Unique labels in a batch:", torch.unique(batch["labels"]))

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

Label dtype: torch.int64
Unique labels in a batch: tensor([0, 1])


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
y = train_df["label_bin"].values
n0 = (y == 0).sum()
n1 = (y == 1).sum()

w0 = 1.0
w1 = n0 / max(n1, 1)

class_weights = torch.tensor([w0, w1], dtype=torch.float32)
print("n0:", n0, "n1:", n1)
print("class_weights:", class_weights)

n0: 7581 n1: 794
class_weights: tensor([1.0000, 9.5479])


In [7]:


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = (probs[:, 1] >= 0.5).astype(int)

    f1_pos = f1_score(labels, preds, pos_label=1)
    prec   = precision_score(labels, preds, pos_label=1, zero_division=0)
    rec    = recall_score(labels, preds, pos_label=1, zero_division=0)
    return {"f1_pos": f1_pos, "precision_pos": prec, "recall_pos": rec}

class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"].long()
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits

        # match device + dtype
        weights = self.class_weights.to(device=logits.device, dtype=logits.dtype)

        loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

args = TrainingArguments(
    output_dir="bestmodel_deberta",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=1.5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pos",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=42,
    fp16=False,
    bf16=False
)

trainer = WeightedLossTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [8]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Pos,Precision Pos,Recall Pos
1,0.477759,0.415896,0.486172,0.342213,0.839196
2,0.409026,0.378019,0.513821,0.379808,0.793970
3,0.248385,0.359783,0.569767,0.463722,0.738693
4,0.208444,0.402175,0.580000,0.481728,0.728643


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1048, training_loss=0.34580965351512416, metrics={'train_runtime': 916.1595, 'train_samples_per_second': 36.566, 'train_steps_per_second': 1.144, 'total_flos': 4407110177280000.0, 'train_loss': 0.34580965351512416, 'epoch': 4.0})

In [9]:
pred = trainer.predict(dev_tok)
logits = pred.predictions
labels = pred.label_ids

probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]

thresholds = np.linspace(0.05, 0.95, 19)
best_t, best_f1 = 0.5, -1

for t in thresholds:
    yhat = (probs >= t).astype(int)
    f1 = f1_score(labels, yhat, pos_label=1)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print("Best threshold:", best_t, "Best dev F1(pos):", best_f1)

# show report at best threshold
yhat_best = (probs >= best_t).astype(int)
print(classification_report(labels, yhat_best, digits=4))

Best threshold: 0.9 Best dev F1(pos): 0.6034912718204489
              precision    recall  f1-score   support

           0     0.9588    0.9573    0.9580      1895
           1     0.5990    0.6080    0.6035       199

    accuracy                         0.9241      2094
   macro avg     0.7789    0.7826    0.7808      2094
weighted avg     0.9246    0.9241    0.9243      2094



In [10]:
# Ensure par_id types match dev_ids
dev_ids_int = pd.Index(pd.Series(dev_ids).astype(int))

tmp = df.loc[df["par_id"].astype(int).isin(dev_ids_int), ["par_id", "text", "label_bin"]].copy()
tmp["par_id"] = tmp["par_id"].astype(int)

# Find missing ids
have = pd.Index(tmp["par_id"])
missing = dev_ids_int.difference(have)
print("Missing par_ids:", list(missing)[:20], "..." if len(missing) > 20 else "")
print("Total missing:", len(missing))

# Reindex in exact order, dropping missing
dev_ordered = (
    tmp.set_index("par_id")
       .reindex(dev_ids_int)
       .dropna(subset=["text"]) 
       .reset_index()
)

# Tokenize + predict
dev_out_ds = Dataset.from_pandas(dev_ordered[["text"]], preserve_index=False)
dev_out_tok = dev_out_ds.map(tokenize_batch, batched=True)
dev_out_tok = dev_out_tok.remove_columns(["text"])
dev_out_tok.set_format("torch")

pred2 = trainer.predict(dev_out_tok)
probs2 = torch.softmax(torch.tensor(pred2.predictions), dim=-1).numpy()[:, 1]
dev_preds = (probs2 >= best_t).astype(int)

# Save
with open("dev.txt", "w") as f:
    for p in dev_preds:
        f.write(f"{int(p)}\n")

print("Wrote dev.txt with", len(dev_preds), "lines")

Missing par_ids: [] 
Total missing: 0


Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

Wrote dev.txt with 2094 lines


In [11]:


TEST_TSV = "/kaggle/input/datasets/temirlansergazin/datasettest/task4_test.tsv"

# Test file has 5 columns: id, art_id, keyword, country, text
test_df = pd.read_csv(
    TEST_TSV,
    sep="\t",
    header=None,
    names=["test_id", "art_id", "keyword", "country", "text"],
    engine="python",
    quoting=3,
    on_bad_lines="skip"
)

# Clean text exactly like training
def strip_html(text: str) -> str:
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

test_df["text"] = test_df["text"].apply(strip_html)

print("Test shape:", test_df.shape)
test_df.head()

Test shape: (3832, 5)


,test_id,art_id,keyword,country,text
0,t_0,@@7258997,vulnerable,us,"In the meantime , conservatives are working to..."
1,t_1,@@16397324,women,pk,In most poor households with no education chil...
2,t_2,@@16257812,migrant,ca,The real question is not whether immigration i...
3,t_3,@@3509652,migrant,gb,"In total , the country 's immigrant population..."
4,t_4,@@477506,vulnerable,ca,"Members of the church , which is part of Ken C..."


In [12]:
from datasets import Dataset
import numpy as np
import torch

best_t = 0.55 

test_ds = Dataset.from_pandas(test_df[["text"]].reset_index(drop=True))

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

test_tok = test_ds.map(tokenize_batch, batched=True)
test_tok = test_tok.remove_columns(["text"])
test_tok.set_format("torch")

# Predict
pred = trainer.predict(test_tok)
logits = pred.predictions
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
test_preds = (probs >= best_t).astype(int)

# Write to test.txt
OUT_PATH = "test.txt"
with open(OUT_PATH, "w") as f:
    for p in test_preds:
        f.write(f"{int(p)}\n")

print(f"Wrote {OUT_PATH} with {len(test_preds)} lines.")
print("First 25 preds:", test_preds[:25])
print("Positive rate:", test_preds.mean())

Map:   0%|          | 0/3832 [00:00<?, ? examples/s]

Wrote test.txt with 3832 lines.
First 25 preds: [0 1 0 0 0 1 1 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 1 1]
Positive rate: 0.11821503131524008
